# Synthing: a TB-303 demo

A follow-up to [18-filters](18-filters.ipynb), where we built a ladder filter from scratch. Here it plays the part it was made for: Jupylet's replica of the Roland TB-303.

Roland's [TB-303](https://en.wikipedia.org/wiki/Roland_TB-303) Bass Line came out in 1981, meant to replace the bass guitar. It sounded nothing like one, sold poorly, and was gone by 1984, a flop. A few years later in Chicago, three friends who called themselves [Phuture](https://en.wikipedia.org/wiki/Phuture) bought one second hand and jammed with it, turning its filter knobs by hand. Their [*Acid Tracks*](https://en.wikipedia.org/wiki/Acid_Tracks) (1987) started [acid house](https://en.wikipedia.org/wiki/Acid_house), and turned the failed bass machine into one of the most sought after synthesizers in dance music. [Have a listen](https://www.youtube.com/watch?v=igNBeo3QSqc).

<a href="https://en.wikipedia.org/wiki/Roland_TB-303"><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/fd/Roland_TB-303_Panel.jpg/960px-Roland_TB-303_Panel.jpg" alt="Roland TB-303" width="560"></a>

*The TB-303. Photo: Steve Sims, CC0, via
[Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Roland_TB-303_Panel.jpg).*

Run all the cells. The bass line starts playing; then turn the knobs on the front panel while it plays, and watch the oscilloscope follow the sound. Edit the pattern and run its cell again: the new one takes over at the end of the current cycle, in time. `/a` marks an accent, `/s` a slide, `.` a rest and `-` a held note.

To learn how the TB-303 works, and how this replica models it, read the introduction to `jupylet/audio/synth/tb303_bassline.py`. For more on sound in Jupylet, see the [Programming Sound and Music](https://jupylet.readthedocs.io/en/latest/programmers_reference_guide/sound.html) section of the Jupylet Programmer's Reference Guide.

In [36]:
import logging
import asyncio
import math
import sys
import os

In [37]:
sys.path.insert(0, os.path.abspath('./..'))

In [38]:
from jupylet.audio.bundle import *

In [39]:
from jupylet.app import App
from jupylet.shadertoy import Shadertoy, get_shadertoy_audio

In [40]:
app = App(width=640, height=256)

In [41]:
canvas = app.run(1/24)

The next cell holds a little program written in GLSL, the OpenGL Shading Language, which runs on your graphics card. It draws the sound as it plays: on the canvas below the front panel, you will see the wave moving, as on an [oscilloscope](https://en.wikipedia.org/wiki/Oscilloscope), an instrument that shows how a signal changes over time. Programs like it are called Shadertoys (see the [Shadertoys](https://jupylet.readthedocs.io/en/latest/programmers_reference_guide/graphics-3d.html#shadertoys) section of the Jupylet Programmer's Reference Guide). Curious how it draws the wave? Ask Claude to walk you through it.

In [42]:
scope = Shadertoy( """

    void mainImage( out vec4 fragColor, in vec2 fragCoord )
    {
        // Normalized pixel coordinates (from 0 to 1)
        vec2 uv = fragCoord / iResolution.xy;

        // Time varying pixel color
        vec3 col = 0.2 + 0.2 * cos(iTime + uv.xyx + vec3(0, 2, 4));

        float amp = texture(iChannel0, vec2(uv.x, 1.)).r; 

        vec3 sig = vec3(0.00033 / max(pow(amp - uv.y, 2.), 1e-6));

        sig *= vec3(.5, .5, 4.) / 2.;

        col += sig;

        // Output to screen
        fragColor = vec4(col,1.0);
    }
    
""", 640, 256, 0, 256, 0, 'left', 'top')

In [43]:
@app.event
def render(ct, dt):
    scope.set_channel(0, *get_shadertoy_audio(amp=5, length=1200))
    scope.render(ct, dt)

In [44]:
tb303 = TB303(
    waveform='sawtooth',
    tuning=0.5,
    cutoff=0.8, 
    resonance=0.75, 
    env_mod=0.2, 
    decay=0.4, 
    accent=0.5, 
    amp=0.75,
    quality='quality',
)

In [45]:
panel = get_tb303_panel(tb303)
panel

Panel(children=(VBox(children=(Label(value='WAVEFORM'), ToggleButtons(options=(('SAW', 'sawtooth'), ('SQUARE',…

In [11]:
canvas

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x05\x03\x0…

In [46]:
set_bpm(96)

In [47]:
tb303.sonic_live_loop("""

    A1/a  .     A2    A1
    A1/a  A1    .     A2
    A1/a  .     C2    A1
    A1/a  E2/s  A1    G1

""", step_duration=1/16)

Knobs can also be turned from code, in time with the pattern. The first cell below opens the cutoff, the resonance and the env mod all the way. The next one sweeps each of them down to 0 and back, as a sine wave, each at its own pace: the cutoff every 3 cycles, the env mod every 5, and the resonance every 2. They drift in and out of step with each other, so the sound keeps changing.

`sweep_now()` starts on the pattern's next step, and `sweep()` at the start of its next cycle; their docstrings tell the rest.

In [50]:
tb303.resonance = 1
tb303.env_mod = 1
tb303.cutoff = 1

In [53]:
tb303.sweep_now('cutoff', 0., 3, shape='sine', repeat=6)
tb303.sweep_now('env_mod', 0., 5, shape='sine', repeat=4)
tb303.sweep_now('resonance', 0., 2, shape='sine', repeat=10)

In [35]:
#tb303.stop()